# Image Classification Validation (FlashSequential)

Validate a `FlashSequential` image classifier against an existing validation iterator.
Run this notebook below a training notebook, or set `model_path` and `validation_dir` to work standalone.

In [ ]:
# Leave these as None when attaching this notebook below a training notebook.
model_path = None  # optional path to a saved Keras model
validation_dir = None  # optional directory with one subfolder per class
image_size = (128, 128)
batch_size = 32
color_mode = "rgb"

## 1. Use the model and validation data

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from flashkeras import FlashSequential
from flashkeras.data_collecting.images import load_all_classes_from_directory_and_preprocess

if model_path is not None:
    model = FlashSequential("classification")
    model.loadModel(model_path)
elif "model" not in globals():
    raise NameError("Define `model` in a previous cell or set `model_path`.")

if validation_dir is not None:
    val_ds = load_all_classes_from_directory_and_preprocess(
        validation_dir,
        batch_size=batch_size,
        img_shape=image_size,
        color_mode=color_mode,
    )
elif "val_ds" not in globals():
    raise NameError("Define `val_ds` in a previous cell or set `validation_dir`.")

print(f"Validation samples: {val_ds.samples}")
print(f"Classes: {val_ds.class_indices}")

## 2. Evaluate metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

val_ds.reset()
evaluation = model.model.evaluate(val_ds, return_dict=True, verbose=0)
val_ds.reset()
probabilities = model.predict(val_ds, verbose=0)
if probabilities.ndim == 2 and probabilities.shape[1] == 1:
    predicted_classes = (probabilities[:, 0] >= 0.5).astype(int)
else:
    predicted_classes = np.argmax(probabilities, axis=1)
true_classes = val_ds.classes
class_names = [name for name, _ in sorted(val_ds.class_indices.items(), key=lambda item: item[1])]

print("Keras evaluation:")
for metric_name, value in evaluation.items():
    print(f"{metric_name}: {value:.4f}")
print("\nClassification report:")
print(classification_report(true_classes, predicted_classes, target_names=class_names, zero_division=0))

ConfusionMatrixDisplay(
    confusion_matrix(true_classes, predicted_classes),
    display_labels=class_names,
).plot(xticks_rotation=45)
plt.tight_layout()